# ds005170 (Chisco) dataset audit

This notebook inspects the raw BIDS structure, EDF annotations, and Chinese text-label spreadsheets. It does **not** preprocess, alter, or download data.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
from IPython.display import display

def locate_dataset(start: Path) -> Path:
    for parent in (start, *start.parents):
        candidate = parent / 'data' / 'ds005170'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/ds005170. Open this notebook inside eeg_to_voice_chinese.')

DATASET = locate_dataset(Path.cwd().resolve())
description = json.loads((DATASET / 'dataset_description.json').read_text())
print(f'DATASET = {DATASET}')
print(description['Name'])

## 1. BIDS inventory

The raw recordings are continuous EDF files. They have EDF annotations but no BIDS `events.tsv`; text labels live in `textdataset/`.

In [ ]:
edf_files = sorted(DATASET.glob('sub-*/ses-*/eeg/*_eeg.edf'))
text_files = sorted(
    (DATASET / 'textdataset').glob('split_data_*.xlsx'),
    key=lambda p: int(re.search(r'(\d+)$', p.stem).group(1)),
)
subjects = sorted({path.parts[-4] for path in edf_files})
sessions = sorted({path.parts[-3] for path in edf_files})

display(pd.Series({
    'subjects': len(subjects),
    'raw EDF recordings': len(edf_files),
    'runs per subject': len(edf_files) // len(subjects),
    'text-label spreadsheets': len(text_files),
    'BIDS events.tsv files': len(list(DATASET.glob('sub-*/ses-*/eeg/*_events.tsv'))),
    'audio WAV files': len(list(DATASET.rglob('*.wav'))),
}).to_frame('count'))

run_inventory = []
for path in edf_files:
    match = re.search(r'(sub-\d+)/(ses-\d+)/eeg/.*run-(\d+)_eeg\.edf$', path.as_posix())
    run_inventory.append({'subject': match.group(1), 'session': match.group(2), 'run': int(match.group(3))})
run_inventory = pd.DataFrame(run_inventory)
display(run_inventory.groupby(['subject', 'session']).size().rename('n_runs').unstack(fill_value=0))

## 2. Read the spreadsheet labels without external Excel packages

The helper below reads `xlsx` directly from its XML container, so it works even when `openpyxl` is not installed. Each row represents one prompted Chinese sentence and its semantic label.

In [ ]:
XLSX_NS = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def read_two_column_xlsx(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as archive:
        xml = archive.read('xl/worksheets/sheet1.xml')
    sheet = ET.fromstring(xml)
    rows = []
    for row in sheet.findall('.//x:sheetData/x:row', XLSX_NS):
        values = {}
        for cell in row.findall('x:c', XLSX_NS):
            column = re.match(r'[A-Z]+', cell.attrib['r']).group()
            values[column] = ''.join(node.text or '' for node in cell.findall('.//x:t', XLSX_NS))
        rows.append(values)
    header = rows[0]
    return pd.DataFrame([{
        header.get('A', 'sentence'): row.get('A', ''),
        header.get('B', 'label'): row.get('B', ''),
    } for row in rows[1:]])

label_frames = []
for path in text_files:
    frame = read_two_column_xlsx(path)
    frame['run'] = int(re.search(r'(\d+)$', path.stem).group(1))
    frame['source_file'] = path.name
    label_frames.append(frame)

labels = pd.concat(label_frames, ignore_index=True).rename(columns={'句子': 'sentence', '标签': 'semantic_label'})
display(labels.head())
display(labels.groupby('run').size().rename('n_prompted_sentences').to_frame().T)

## 3. Trial and label counts

The spreadsheet schedule has 45 runs. Every subject has the same run numbers `1…45`, so the primary audit estimate is the schedule total multiplied by three subjects. The EDF-header cell below validates representative run-level annotation counts.

In [ ]:
n_schedule_trials = len(labels)
n_unique_sentences = labels['sentence'].nunique()
n_semantic_labels = labels['semantic_label'].nunique()
n_subjects = len(subjects)

display(pd.Series({
    'text schedule rows / trials per subject': n_schedule_trials,
    'estimated EEG trials across all subjects': n_schedule_trials * n_subjects,
    'unique sentence strings': n_unique_sentences,
    'semantic labels': n_semantic_labels,
    'raw runs per subject': len(run_inventory) // n_subjects,
}).to_frame('count'))

display(labels['semantic_label'].value_counts().rename_axis('semantic_label').to_frame('n_schedule_trials'))

## 4. Cross-check text categories against the supplied JSON mapping

In [ ]:
text_to_class = json.loads((DATASET / 'json' / 'textmaps.json').read_text())
class_to_name = json.loads((DATASET / 'json' / 'classnumber.json').read_text())

mapped_codes = labels['sentence'].map(text_to_class)
print('Rows with a textmaps.json code:', mapped_codes.notna().sum(), '/', len(labels))
print('Text labels represented in classnumber.json:', labels.semantic_label.isin(class_to_name.values()).sum(), '/', len(labels))
display(pd.DataFrame({
    'class_code': [int(code) for code in sorted(class_to_name, key=int)],
    'semantic_label': [class_to_name[code] for code in sorted(class_to_name, key=int)],
}))

## 5. Optional: inspect EDF headers and annotation counts

This reads only EDF headers (`preload=False`), never the full signal. It checks run 1, a typical 150-sentence run, and the final short run.

In [ ]:
import mne

def edf_path(subject: str, run: int) -> Path:
    matches = sorted(DATASET.glob(f'{subject}/ses-*/eeg/*run-{run:02d}_eeg.edf'))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one EDF for {subject} run {run}; found {len(matches)}')
    return matches[0]

header_rows = []
for run in (1, 2, 45):
    path = edf_path('sub-01', run)
    raw = mne.io.read_raw_edf(path, preload=False, verbose='ERROR')
    header_rows.append({
        'run': run,
        'label_rows': int((labels['run'] == run).sum()),
        'EDF_annotations': len(raw.annotations),
        'sampling_hz': raw.info['sfreq'],
        'channels': len(raw.ch_names),
        'duration_s': round(raw.times[-1], 3),
        'annotation_descriptions': sorted(set(raw.annotations.description)),
    })
display(pd.DataFrame(header_rows))

## 6. Full annotation-count audit (optional)

Set `RUN_FULL_HEADER_AUDIT = True` only when you want to scan annotations in every EDF. It may take several minutes but does not load full EEG arrays.

In [ ]:
RUN_FULL_HEADER_AUDIT = False

if RUN_FULL_HEADER_AUDIT:
    rows = []
    for path in edf_files:
        match = re.search(r'(sub-\d+)/(ses-\d+)/eeg/.*run-(\d+)_eeg\.edf$', path.as_posix())
        raw = mne.io.read_raw_edf(path, preload=False, verbose='ERROR')
        rows.append({
            'subject': match.group(1), 'session': match.group(2), 'run': int(match.group(3)),
            'annotations': len(raw.annotations),
        })
    annotation_audit = pd.DataFrame(rows)
    display(annotation_audit.groupby('subject').annotations.agg(['count', 'sum', 'min', 'max']))
    expected = labels.groupby('run').size().rename('expected_annotations')
    checked = annotation_audit.join(expected, on='run')
    print('Run-level annotation / text-schedule agreement:', (checked.annotations == checked.expected_annotations).mean())
else:
    print('Skipped. Change RUN_FULL_HEADER_AUDIT to True to validate every EDF header.')

## Interpretation before EEG-to-voice modelling

- This is an imagined-speech EEG + Chinese text/semantic-label dataset; no paired WAV recordings are supplied.
- The reconstruction target can therefore be text/content or a separately generated canonical TTS signal, not the participant's original trial-specific voice.
- Build the EEG→sentence/semantic-content manifest only after confirming that EDF annotation order and `split_data_<run>.xlsx` order agree for every run.
- Keep subject-level splits: the same 45 text schedules appear to be reused across the three subjects, so random trial splitting can overestimate cross-subject generalization.